In [1]:
import os
import glob
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops

from sklearn.metrics import cohen_kappa_score
import timm

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Using device: cuda:0


In [2]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = torch.zeros(out.size(0))
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu()
    return prediction


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(out[i]))
            l2 = int(math.ceil(out[i]))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob




In [3]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()
        self.backbone = timm.models.tf_efficientnet_b5_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None, backbone_pretrained=False):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b4_ns(
            pretrained=backbone_pretrained
        )
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [4]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]
        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)
        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w

        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h
        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image




In [5]:
def resolve_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


DATA_DIR = resolve_existing_path(
    [
        "../input/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection",
        "/kaggle/data/aptos2019-blindness-detection",
    ]
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate aptos2019-blindness-detection dataset directory."
    )

TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")

test_df = pd.read_csv(TEST_CSV)
test_ids = test_df["id_code"].astype(str).values

train_df = pd.read_csv(TRAIN_CSV)
y = train_df["diagnosis"].astype(int).values

try:
    class_counts = np.bincount(y, minlength=5).astype(float)
    priors = class_counts / max(class_counts.sum(), 1.0)
    print("Train class priors:", priors.round(4).tolist())
except Exception as e:
    priors = np.array([1, 0, 0, 0, 0], dtype=float)
    print("Could not compute priors; using fallback. Reason:", repr(e))

class_counts_int = np.bincount(y, minlength=5).astype(int)
print("Train class counts:", class_counts_int.tolist())

input_size = 380
transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

weights_dir = resolve_existing_path(
    ["../input/weights", "/kaggle/input/weights", "/kaggle/data/weights"]
)
weights_path = None
if weights_dir is not None:
    preferred = os.path.join(weights_dir, "B4_3stage_49epoch_CLAHE.pkl")
    if os.path.exists(preferred):
        weights_path = preferred
    else:
        candidates = (
            glob.glob(os.path.join(weights_dir, "*.pkl"))
            + glob.glob(os.path.join(weights_dir, "*.pth"))
            + glob.glob(os.path.join(weights_dir, "*.pt"))
        )
        candidates = sorted(candidates)
        if len(candidates) > 0:
            weights_path = candidates[0]


def find_local_dr_checkpoint():
    search_roots = [
        "/kaggle/input",
        "/kaggle/data",
        "../input",
        DATA_DIR,
    ]
    exts = (".pth", ".pt", ".bin")
    key_hints = ("aptos", "blind", "retina", "dr", "diabetic", "efficientnet", "b4")
    hits = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for ext in exts:
            pattern = os.path.join(root, "**", f"*{ext}")
            for p in glob.glob(pattern, recursive=True):
                name = os.path.basename(p).lower()
                if any(k in name for k in key_hints):
                    try:
                        sz = os.path.getsize(p)
                    except Exception:
                        sz = 0
                    if sz > 1_000_000:  # >1MB
                        hits.append((p, sz))
    hits = sorted(hits, key=lambda x: -x[1])
    return hits[0][0] if hits else None


fallback_ckpt = None
if weights_path is None:
    fallback_ckpt = find_local_dr_checkpoint()
    if fallback_ckpt is not None:
        print("Found local DR checkpoint candidate for fallback:", fallback_ckpt)

use_pretrained_backbone = weights_path is None
net = ThreeStage_Model(backbone_pretrained=use_pretrained_backbone)

if weights_path is not None:
    print("Loading weights:", weights_path)
    state = torch.load(weights_path, map_location="cpu")
    if (
        isinstance(state, dict)
        and "state_dict" in state
        and isinstance(state["state_dict"], dict)
    ):
        state = state["state_dict"]
    try:
        net.load_state_dict(state, strict=True)
    except RuntimeError:
        new_state = {}
        for k, v in state.items():
            nk = k
            if nk.startswith("module."):
                nk = nk[len("module.") :]
            new_state[nk] = v
        net.load_state_dict(new_state, strict=False)
else:
    print(
        "WARNING: No competition weights found; using ImageNet-pretrained backbone. Using train-set prototypes for fallback."
    )
    if fallback_ckpt is not None:
        try:
            st = torch.load(fallback_ckpt, map_location="cpu")
            if (
                isinstance(st, dict)
                and "state_dict" in st
                and isinstance(st["state_dict"], dict)
            ):
                st = st["state_dict"]

            cleaned = {}
            if isinstance(st, dict):
                for k, v in st.items():
                    nk = k
                    if nk.startswith("module."):
                        nk = nk[len("module.") :]
                    cleaned[nk] = v

            missing, unexpected = net.load_state_dict(cleaned, strict=False)
            print(
                "Loaded fallback checkpoint (non-strict). Missing:",
                len(missing),
                "Unexpected:",
                len(unexpected),
            )
        except Exception as e:
            print(
                "Could not load fallback checkpoint; continuing with ImageNet pretrained. Reason:",
                repr(e),
            )

net = net.to(device)
net.eval()

USE_PROTOTYPE_FALLBACK = weights_path is None


def _extract_backbone_feature(net, x):
    # Change (score-relevant, minimal): use a consistent "pre-classifier embedding"
    # for prototypes/whitening/inference. Avoid calling forward_head(pre_logits=True)
    # on an already pooled tensor (which can be shape/semantics-mismatched in timm).
    f = net.backbone.forward_features(x)  # [B,C,H,W]
    f = net.backbone.global_pool(f)  # GeM -> [B,C] (flatten=True)
    if f.ndim > 2:
        f = f.flatten(1)
    return f  # stable embedding used everywhere below


def _fit_whitener(feats_cpu: torch.Tensor, eps: float = 1e-5, shrink: float = 0.10):
    feats_cpu = feats_cpu.float()
    feats_cpu = feats_cpu / (feats_cpu.norm(p=2, dim=1, keepdim=True) + 1e-12)

    mu = feats_cpu.mean(dim=0)
    xc = feats_cpu - mu
    cov = (xc.t() @ xc) / max(xc.shape[0] - 1, 1)
    d = cov.shape[0]

    diag = torch.diag(torch.diag(cov))
    cov = (1.0 - shrink) * cov + shrink * diag
    cov = cov + eps * torch.eye(d, dtype=cov.dtype)

    w, v = torch.linalg.eigh(cov)
    w = torch.clamp(w, min=eps)
    inv_sqrt = (v * (1.0 / torch.sqrt(w)).unsqueeze(0)) @ v.t()
    return mu, inv_sqrt


def build_class_prototypes_and_whitener(
    net,
    train_df,
    img_dir,
    transform,
    device,
    max_per_class=260,
    seed=0,
):
    rng = np.random.RandomState(seed)
    df = train_df.copy()
    df["diagnosis"] = df["diagnosis"].astype(int)

    selected_rows = []
    for cls in range(5):
        cls_rows = df[df["diagnosis"] == cls]
        if len(cls_rows) == 0:
            continue
        take = min(max_per_class, len(cls_rows))
        idxs = rng.choice(len(cls_rows), size=take, replace=False)
        selected_rows.append(cls_rows.iloc[idxs])
    sel = pd.concat(selected_rows, axis=0).reset_index(drop=True)

    feats_by_class = {c: [] for c in range(5)}
    all_feats = []
    with torch.no_grad():
        for i in range(len(sel)):
            img_id = str(sel.loc[i, "id_code"])
            cls = int(sel.loc[i, "diagnosis"])
            path = os.path.join(img_dir, f"{img_id}.png")
            try:
                img = Image.open(path).convert("RGB")
                x = transform(img).unsqueeze(0).to(device)
                f = _extract_backbone_feature(net, x)  # [1,D]
                f = f.detach().float().cpu().squeeze(0)  # [D]
                feats_by_class[cls].append(f)
                all_feats.append(f)
            except Exception:
                continue

    if len(all_feats) < 50:
        feat_dim = int(all_feats[0].numel()) if len(all_feats) > 0 else 1792
        mu = torch.zeros(feat_dim, dtype=torch.float32)
        inv_sqrt = torch.eye(feat_dim, dtype=torch.float32)
    else:
        feats_mat = torch.stack(all_feats, dim=0)  # [N,D]
        mu, inv_sqrt = _fit_whitener(feats_mat, eps=1e-4, shrink=0.10)

    def whiten_and_norm(f_cpu_1d: torch.Tensor):
        fw = (f_cpu_1d - mu) @ inv_sqrt.t()
        fw = fw / (fw.norm(p=2) + 1e-12)
        return fw

    protos = []
    for cls in range(5):
        if len(feats_by_class[cls]) == 0:
            protos.append(torch.zeros_like(mu))
        else:
            stack = torch.stack(
                [whiten_and_norm(z) for z in feats_by_class[cls]], dim=0
            )
            m = stack.mean(dim=0)
            m = m / (m.norm(p=2) + 1e-12)
            protos.append(m)
    protos = torch.stack(protos, dim=0)  # [5,D]

    return protos.to(device), mu.to(device), inv_sqrt.to(device)


def compute_severity_from_image(
    net,
    img,
    transform,
    device,
    prototypes,
    mu,
    inv_sqrt,
):
    x = transform(img).unsqueeze(0).to(device)
    f = _extract_backbone_feature(net, x)  # [1,D]
    f = (f - mu.unsqueeze(0)) @ inv_sqrt.t()
    f = F.normalize(f, dim=1)
    sims = torch.matmul(f, prototypes.t()).squeeze(0)  # [5]
    probs = F.softmax(sims, dim=0)
    proto_labels = torch.arange(5, device=device, dtype=torch.float32)
    sev = (probs * proto_labels).sum()
    return sev


def optimize_thresholds_qwk(
    sevs: np.ndarray,
    y_true: np.ndarray,
    init_thr: list,
    n_iter: int = 2,
    grid_points: int = 25,
):
    sevs = sevs.astype(np.float64)
    y_true = y_true.astype(int)

    def apply_thr(vals, thr):
        thr0, thr1, thr2, thr3 = thr
        out = np.zeros_like(vals, dtype=np.int64)
        out += (vals >= thr0).astype(np.int64)
        out += (vals >= thr1).astype(np.int64)
        out += (vals >= thr2).astype(np.int64)
        out += (vals >= thr3).astype(np.int64)
        return out

    best_thr = [float(x) for x in init_thr]
    best_k = cohen_kappa_score(y_true, apply_thr(sevs, best_thr), weights="quadratic")

    lo, hi = float(np.min(sevs)), float(np.max(sevs))
    span = max(hi - lo, 1e-6)
    for it in range(n_iter):
        for j in range(4):
            center = best_thr[j]
            radius = span * (0.15 if it == 0 else 0.08)
            cand = np.linspace(center - radius, center + radius, grid_points)

            for v in cand:
                thr = best_thr.copy()
                thr[j] = float(v)
                thr = sorted(thr)
                for k in range(1, 4):
                    if thr[k] <= thr[k - 1]:
                        thr[k] = thr[k - 1] + 1e-4
                kappa = cohen_kappa_score(
                    y_true, apply_thr(sevs, thr), weights="quadratic"
                )
                if kappa > best_k:
                    best_k = kappa
                    best_thr = thr

    return best_thr, float(best_k)


prototypes = None
mu_w = None
inv_sqrt_w = None

if USE_PROTOTYPE_FALLBACK:
    t0 = time.time()
    prototypes, mu_w, inv_sqrt_w = build_class_prototypes_and_whitener(
        net=net,
        train_df=train_df,
        img_dir=TRAIN_IMG_DIR,
        transform=transform,
        device=device,
        max_per_class=260,
        seed=0,
    )
    print("Built prototypes+whitener in %.1fs" % (time.time() - t0))

    t1 = time.time()
    rng = np.random.RandomState(1)
    df_cal = train_df.copy().reset_index(drop=True)
    max_cal = 1200
    if len(df_cal) > max_cal:
        idxs = rng.choice(len(df_cal), size=max_cal, replace=False)
        df_cal = df_cal.iloc[idxs].reset_index(drop=True)

    sevs = []
    y_true = []
    with torch.no_grad():
        for i in range(len(df_cal)):
            img_id = str(df_cal.loc[i, "id_code"])
            y_i = int(df_cal.loc[i, "diagnosis"])
            path = os.path.join(TRAIN_IMG_DIR, f"{img_id}.png")
            try:
                img = Image.open(path).convert("RGB")
                sev_t = compute_severity_from_image(
                    net=net,
                    img=img,
                    transform=transform,
                    device=device,
                    prototypes=prototypes,
                    mu=mu_w,
                    inv_sqrt=inv_sqrt_w,
                )
                sev = float(sev_t.item())
                if math.isfinite(sev):
                    sevs.append(sev)
                    y_true.append(y_i)
            except Exception:
                continue

    if len(sevs) < 80:
        print("Threshold calibration skipped: too few train predictions:", len(sevs))
    else:
        sevs_np = np.asarray(sevs, dtype=np.float64)
        y_true_np = np.asarray(y_true, dtype=np.int64)

        order = np.argsort(sevs_np)
        sevs_sorted = sevs_np[order]
        n = len(sevs_sorted)
        cum_priors = np.cumsum(np.asarray(priors, dtype=np.float64))
        init_cuts = []
        for k in range(4):
            q = float(cum_priors[k])
            q = min(max(q, 0.0), 1.0)
            idx = int(round(q * (n - 1)))
            idx = max(0, min(n - 1, idx))
            init_cuts.append(float(sevs_sorted[idx]))
        for i in range(1, 4):
            if init_cuts[i] <= init_cuts[i - 1]:
                init_cuts[i] = init_cuts[i - 1] + 1e-3

        tuned_thr, tuned_k = optimize_thresholds_qwk(
            sevs=sevs_np,
            y_true=y_true_np,
            init_thr=init_cuts,
            n_iter=2,
            grid_points=23,
        )
        threshold = [float(x) for x in tuned_thr]
        print(
            "Calibrated thresholds (fallback, QWK-tuned):",
            threshold,
            "cal_kappa=%.5f" % tuned_k,
            "in %.1fs" % (time.time() - t1),
        )

print("Final thresholds used by regress2class:", threshold)



Train class priors: [0.4941, 0.1032, 0.2719, 0.0534, 0.0774]
Train class counts: [1628, 340, 896, 176, 255]


/tmp/ipykernel_55/1588415685.py:49: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Built prototypes+whitener in 199.8s
Calibrated thresholds (fallback, QWK-tuned): [1.999914288520813, 2.0001577845486724, 2.0005253271623094, 2.00095204223286] cal_kappa=0.71581 in 152.9s
Final thresholds used by regress2class: [1.999914288520813, 2.0001577845486724, 2.0005253271623094, 2.00095204223286]


In [6]:
submission = []

proto_labels = (
    torch.arange(5, device=device, dtype=torch.float32)
    if USE_PROTOTYPE_FALLBACK
    else None
)

with torch.no_grad():
    for i, idx in enumerate(test_ids):
        if i % 50 == 0:
            print("Processed", i, "/", len(test_ids))

        image_name = os.path.join(TEST_IMG_DIR, f"{idx}.png")
        try:
            img = Image.open(image_name).convert("RGB")
            img_t = transform(img).unsqueeze(0).to(device)

            if USE_PROTOTYPE_FALLBACK:
                f = _extract_backbone_feature(net, img_t)  # [1,D]
                f = (f - mu_w.unsqueeze(0)) @ inv_sqrt_w.t()
                f = F.normalize(f, dim=1)
                sims = torch.matmul(f, prototypes.t()).squeeze(0)  # [5]
                probs = F.softmax(sims, dim=0)  # [5]
                sev = (probs * proto_labels).sum().view(1)  # [1]
                pred_class = int(regress2class(sev)[0].item())
                submission.append([idx, pred_class])
            else:
                c_out, r_out, _ = net(img_t)
                r_out_1d = r_out.view(-1)
                pred = regress2class(r_out_1d)
                pred_class = int(pred.item())
                submission.append([idx, pred_class])
        except Exception:
            submission.append([idx, 0])

submission = np.array(submission, dtype=object)



Processed 0 / 367
Processed 50 / 367
Processed 100 / 367
Processed 150 / 367
Processed 200 / 367
Processed 250 / 367
Processed 300 / 367
Processed 350 / 367


In [7]:
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])

if df.shape[0] != len(test_ids):
    raise RuntimeError(
        f"Submission row count mismatch: got {df.shape[0]}, expected {len(test_ids)}"
    )

df["id_code"] = df["id_code"].astype(str)
df["diagnosis"] = df["diagnosis"].astype(int)
df = test_df.merge(df, on="id_code", how="left", validate="one_to_one")
if df["diagnosis"].isna().any():
    df["diagnosis"] = df["diagnosis"].fillna(0).astype(int)

df.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", df.shape)
print(df.head())

Wrote submission.csv with shape: (367, 2)
        id_code  diagnosis
0  b460ca9fa26f          2
1  6cee2e148520          0
2  ca6842bfcbc9          0
3  6cbc3dad809c          2
4  a9bc2f892cb3          0
